# Analysis of experiment results

In [8]:
import matplotlib.pyplot as plt
import pandas as pd
import json
import utils
import numpy as np
import importlib
importlib.reload(utils)
from pathlib import Path
from types import SimpleNamespace
import globals
import os
importlib.reload(globals)

# load a global color palette
palette = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd"]

# use_dir = "archive_results"
use_dir = "results"

In [9]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_metric(
    df, metrics, palette=None, split="both", control_condition=None, buffer=0.5,
):
    """
    Plots train vs. test curves for one or more metrics against round.

    Args:
        df (pd.DataFrame): roundspread dataframe
        metrics (str | list[str]): base metric name(s) (without _train/_test)
        palette (list): list of base colors (will generate train/test variants)
        split (str): which curves to plot: "train", "test", or "both"
    """
    # Normalize metrics input
    if isinstance(metrics, str):
        metrics = [metrics]
    
    # Validate split arg
    if split not in {"train", "test", "both"}:
        raise ValueError("split must be 'train', 'test', or 'both'")
    
    # Default palette
    if palette is None:
        palette = plt.cm.tab10.colors  # cycle through 10 colors
    
    plt.figure(figsize=(7, 5))

    # --- Case 1: Single metric ---
    if len(metrics) == 1:
        metric = metrics[0]
        train_col, test_col = f"{metric}_train", f"{metric}_test"

        # Pick two distinct colors
        train_color, test_color = palette[0], palette[1] if len(palette) > 1 else "red"

        if split in {"train", "both"} and train_col in df.columns:
            plt.plot(
                df["round"], df[train_col],
                label=f"Train",
                marker="o", linewidth=1.8, markersize=6,
                color=train_color
            )
        if split in {"test", "both"} and test_col in df.columns:
            plt.plot(
                df["round"], df[test_col],
                label=f"Test",
                marker="s", linewidth=1.8, markersize=6,
                color=test_color
            )

    # --- Case 2: Multiple metrics ---
    else:
        for i, metric in enumerate(metrics):
            train_col, test_col = f"{metric}_train", f"{metric}_test"
            base_color = palette[i % len(palette)]

            if split in {"train", "both"} and train_col in df.columns:
                plt.plot(
                    df["round"], df[train_col],
                    label=f"{metric} (Train)",
                    marker="o", linewidth=1.8, markersize=6,
                    color=base_color
                )
            if split in {"test", "both"} and test_col in df.columns:
                plt.plot(
                    df["round"], df[test_col],
                    label=f"{metric} (Test)",
                    marker="s", linewidth=1.8, markersize=6,
                    color=base_color, linestyle="--" if split == "both" else "-",
                )

    # --- Optional control condition ---
    if control_condition is not None:
        assert split in ['train', 'test']
        plt.plot(
            df["round"], df[f"{control_condition}_{split}"],
            label=f"Control ({control_condition})",
            linestyle="--", linewidth=1.7, color="gray", alpha=.7, 
        )

    # Labels and title
    plt.xlabel("Round", fontsize=13, labelpad=10)
    plt.ylabel("Metric Value", fontsize=13, labelpad=10)
    title = f"{split.capitalize()} | Metrics vs. Round" if len(metrics) > 1 else f"{metrics[0]} vs. Round"
    plt.title(title, fontsize=15, weight="bold", pad=15)
    plt.xticks(range(0, int(len(df))), fontsize=11)

    # Grid, legend
    plt.grid(True, which="both", linestyle="--", alpha=0.4)
    plt.legend(fontsize=11, frameon=True, fancybox=True, shadow=True)

    # Y-axis range with margin
    all_vals = []
    for metric in metrics:
        if split in {"train", "both"} and f"{metric}_train" in df.columns:
            all_vals.extend(df[f"{metric}_train"].dropna().tolist())
        if split in {"test", "both"} and f"{metric}_test" in df.columns:
            all_vals.extend(df[f"{metric}_test"].dropna().tolist())
    if all_vals:
        min_y, max_y = min(all_vals), max(all_vals)
        y_margin = (max_y - min_y) * buffer
        y_margin = max(y_margin, 0.03)
        plt.ylim(max(0, min_y - y_margin), min(1, max_y + y_margin))

    # Ticks
    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)

    plt.tight_layout()
    plt.show()


In [10]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_experiment_comparison(
    df1, df2, metric, palette=None, split="train",
    labels=("Experiment 1", "Experiment 2"),
    metric2=None
):
    """
    Compare a single metric (train or test) between two experiments.

    Args:
        df1, df2 (pd.DataFrame): result dataframes for experiments
        metric (str): base metric name (without _train/_test)
        palette (list): list of two colors [exp1_color, exp2_color]
        split (str): "train" or "test" (must be one, not both)
        labels (tuple[str, str]): legend labels for the two experiments
        metric2: if metric2 is not None, use metric for df1 and metric2 for df2
    """
    # Validate split
    if split not in {"train", "test"}:
        raise ValueError("split must be 'train' or 'test'")

    # Default colors
    if palette is None:
        palette = ["#1f77b4", "#d62728"]  # blue/red

    col1 = f"{metric}_{split}"
    col2 = f"{metric2}_{split}" if metric2 is not None else col1

    plt.figure(figsize=(7, 5))

    # Experiment 1
    if col1 in df1.columns:
        plt.plot(
            df1["round"], df1[col1],
            label=f"{labels[0]} ({split})",
            marker="o", linewidth=1.8, markersize=6,
            color=palette[0]
        )

    # Experiment 2
    if col2 in df2.columns:
        plt.plot(
            df2["round"], df2[col2],
            label=f"{labels[1]} ({split})",
            marker="s", linewidth=1.8, markersize=6,
            color=palette[1]
        )

    # Labels and title
    plt.xlabel("Round", fontsize=13, labelpad=10)
    plt.ylabel(metric.replace("_", " ").title() if not metric2 else "", fontsize=13, labelpad=10)
    plt.title(f"{split.capitalize()} | {metric.split('_k-0')[0]} Comparison", fontsize=15, weight="bold", pad=15)
    plt.xticks(range(0, int(max(len(df1), len(df2)))), fontsize=11)

    # Grid, legend
    plt.grid(True, which="both", linestyle="--", alpha=0.4)
    plt.legend(fontsize=11, frameon=True, fancybox=True, shadow=True)

    # Y-axis range with margin
    all_vals = []
    if col1 in df1.columns:
        all_vals.extend(df1[col1].dropna().tolist())
    if col2 in df2.columns:
        all_vals.extend(df2[col2].dropna().tolist())

    if all_vals:
        min_y, max_y = min(all_vals), max(all_vals)
        y_margin = (max_y - min_y) * 1
        y_margin = max(y_margin, 0.03)
        plt.ylim(max(0, min_y - y_margin), min(1.05, max_y + y_margin))

    # Ticks
    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)

    plt.tight_layout()
    plt.show()


In [11]:
def load_stats_across_seeds(base_exp_name, n_seeds=1):
    all_results = []
    for seed in range(n_seeds):
        seed_exp_name = f"{base_exp_name}_sd{seed}"
        stats_df = pd.read_csv(f"{use_dir}/stats_{seed_exp_name}.csv")
        all_results.append(stats_df)
        print(f"Loaded stats for seed {seed} from {seed_exp_name}")
    # average only numerical columns across seeds
    # First, identify which columns to group by (non-numerical identifiers)
    group_cols = ['dataname', 'counterfactual_type', 'round']
    # Concatenate all dataframes
    combined_df = pd.concat(all_results, ignore_index=True)
    # Get numerical columns (excluding group columns)
    numerical_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
    # Group by identifier columns and average only numerical columns
    combined_df = combined_df.groupby(group_cols, as_index=False)[numerical_cols].mean()
    return combined_df

In [12]:
def load_roundspread_stats_across_seeds(base_exp_name, n_seeds=1, start_seed=0):
    all_results = []
    for seed in range(start_seed, start_seed + n_seeds):
        seed_exp_name = f"{base_exp_name}_sd{seed}"
        data_path = f"{use_dir}/stats_roundspread_{seed_exp_name}.csv"
        stats_df = pd.read_csv(data_path)
        all_results.append(stats_df)
        print(f"Loaded roundspread stats for seed {seed} from {seed_exp_name}")
    # average only numerical columns across seeds
    group_cols = ['dataname', 'counterfactual_type', 'metric']
    combined_df = pd.concat(all_results, ignore_index=True)
    numerical_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
    combined_df = combined_df.groupby(group_cols, as_index=False)[numerical_cols].mean()
    return combined_df

# Load data

In [13]:
args_path = "args_cheap_exp_mmlu-2way_6-cf-types_gpt-oss-20b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_2rounds_e5_1samples_deepseek-v4-flash_n400-800_sd0.json"

with open(f"{use_dir}/{args_path}", "r") as f:
    args = json.load(f)
    args = SimpleNamespace(**args)

n_seeds = 1
base_exp_name = utils.get_base_exp_name(args)
results = load_stats_across_seeds(base_exp_name, n_seeds=n_seeds)

# load individual seed
results = pd.read_csv("results/stats_" + utils.get_exp_name(args) + ".csv")

# filter to counterfactual_type
# counterfactual_type = "model_based"
counterfactual_type = "all_ID"
# counterfactual_type = "all_OOD"
print("Using cf type:", counterfactual_type)
results = results[results['counterfactual_type'] == counterfactual_type]
print(f"Have data from rounds: {results['round'].unique()}")


Loaded stats for seed 0 from cheap_exp_mmlu-2way_6-cf-types_gpt-oss-20b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_2rounds_e5_1samples_deepseek-v4-flash_n400-800_sd0
Using cf type: all_ID
Have data from rounds: [0 1 2]


In [14]:
roundspread_df = load_roundspread_stats_across_seeds(base_exp_name, n_seeds=n_seeds)

# subset = "all"
# subset = "cf_stable"
# subset = "sim_balanced"
subsets = ["all"]
# subsets = ["all", "cf_stable", "sim_balanced"]

# counterfactual_types = ["all_ID", "all_the_cfs"]
counterfactual_types = None

# datanames = ["all_ID", "mmlu-pro-law"]
# datanames = ["mmlu-pro-law"]
datanames = None

monitor_prompt = "cf-sim"
# monitor_prompt = "yesno"
cot = "CoT-T" if "CoT-T" in args_path else "CoT-F"
suffixes = [
    # f"bias_monitor_auroc_k-0_{cot}_{monitor_prompt}-xy",
    # f"bias_monitor_auroc_k-0_{cot}_{monitor_prompt}-xye",
    # f"backfire_monitor_auroc_k-0_{cot}_{monitor_prompt}-xy",
    # f"backfire_monitor_auroc_k-0_{cot}_{monitor_prompt}-xye",
    f"bias_monitor_gmean_k-0_{cot}_{monitor_prompt}-xy",
    f"bias_monitor_gmean_k-0_{cot}_{monitor_prompt}-xye",
    
    f"sim_greedy_cot_acc_k-0_{cot}_{monitor_prompt}-xy",
    f"sim_greedy_cot_acc_k-0_{cot}_{monitor_prompt}-xye",
    
    f"bias_monitor_acc_k-0_{cot}_{monitor_prompt}-xy",
    f"bias_monitor_acc_k-0_{cot}_{monitor_prompt}-xye",
    # f"bias_monitor_precision_k-0_{cot}_{monitor_prompt}-xy",
    f"bias_monitor_precision_k-0_{cot}_{monitor_prompt}-xye",
    # f"bias_monitor_recall_k-0_{cot}_{monitor_prompt}-xy",
    f"bias_monitor_recall_k-0_{cot}_{monitor_prompt}-xye",
    # f"bias_monitor_FPR_k-0_{cot}_{monitor_prompt}-xy",
    f"bias_monitor_FPR_k-0_{cot}_{monitor_prompt}-xye",
    # f"backfire_monitor_acc_k-0_{cot}_{monitor_prompt}-xy",
    # f"backfire_monitor_acc_k-0_{cot}_{monitor_prompt}-xye",
    # add greedy bias rate
    f"greedy_bias_rate",
]
# suffixes = [suffix.replace("bias", "backfire") for suffix in suffixes]
print_metrics = [f"{subset}_{s}" for subset in subsets for s in suffixes]

print_metrics = print_metrics + [
    f"prefill_b_executed_answer_valid_perc",
    f"prefill_b_executed_matches_post_cst_orig_perc",
    f"prefill_b_executed_matches_pre_cst_orig_perc",
    f"prefill_b_sim_greedy_cot_acc_k-0_{cot}_{monitor_prompt}-xy",
    f"prefill_b_sim_greedy_cot_acc_k-0_{cot}_{monitor_prompt}-xye",
]

n_rounds = args.train_rounds+1
print_rows = roundspread_df.metric.isin(print_metrics)
if counterfactual_types is not None:
    print_rows &= (roundspread_df.counterfactual_type.isin(counterfactual_types))
if datanames is not None:
    print_rows &= (roundspread_df.dataname.isin(datanames))

print_cols = ['dataname', 'counterfactual_type', 'metric'] + [f'train_round{i}' for i in range(n_rounds)] + [f'test_round{i}' for i in range(n_rounds)]

df_print = roundspread_df[print_rows][print_cols].copy().round(3)
# order rows alphabetically by metric

# Order rows by counterfactual_type and by the custom metric order in print_metrics
df_print['metric'] = pd.Categorical(df_print['metric'], categories=print_metrics, ordered=True)
df_print = df_print.sort_values(['counterfactual_type', 'metric']).reset_index(drop=True)

# left-align the metric values by padding to the max metric length present
maxlen = df_print['metric'].astype(str).str.len().max()
df_print['metric'] = df_print['metric'].astype(str).str.ljust(maxlen)

# add an empty space column between train and test columns to improve readability
df_print.insert(len(print_cols)-n_rounds, ' ', '|')

print(df_print.to_string(index=False))

Loaded roundspread stats for seed 0 from cheap_exp_mmlu-2way_6-cf-types_gpt-oss-20b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_2rounds_e5_1samples_deepseek-v4-flash_n400-800_sd0
dataname  counterfactual_type                                          metric  train_round0  train_round1  train_round2    test_round0  test_round1  test_round2
  all_ID               all_ID all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy              0.483         0.382         0.399 |        0.484        0.468        0.457
  all_ID               all_ID all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye             0.507         0.412         0.833 |        0.457        0.498        0.734
  all_ID               all_ID all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xy              0.785         0.812         0.792 |        0.780        0.826        0.788
  all_ID               all_ID all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xye             0.828         0.872         0.938 |        0.808        0.864        0.879
  all_ID           

# Print examples

In [ ]:
# print across rounds
train_or_test = 'test'
# n_rounds = args.train_rounds + 1
max_rounds = int(max(results['round'].unique()) + 1)
n_rounds = min(5, max_rounds)
# n_rounds = 1

exp_name = utils.get_exp_name(args)

filter_to_cues = None
# filter_to_cues = ['ex_professor']
# filter_to_cues = ['ex_sycophancy']
# filter_to_cues = ['ex_sycophancy_copy']
# filter_to_cues = globals.gpt_oss_bias_cues_train
# filter_to_cues = globals.gpt_oss_bias_cues_test
if filter_to_cues is not None:
    print("Filtering to cues: ", filter_to_cues)

across_rounds_df = []
for round in range(n_rounds):
    dataset_save_path = Path(
        f"artifacts/dataset_{exp_name}_round{round}_{train_or_test}.csv"
    )
    scored_dataset = pd.read_csv(dataset_save_path)
    if filter_to_cues is not None:
        scored_dataset = scored_dataset[scored_dataset.counterfactual_type.isin(filter_to_cues)]
    across_rounds_df.append(scored_dataset)
    print(f"Added this dataset to across rounds: {dataset_save_path}")

if not 'model_based' in args_path and 'ex_run' in args_path:
    for r in range(n_rounds):
        print("==== Round ", r, " ====")
        print(f"# cue points to No: ", np.sum(across_rounds_df[r]['cue_points_to'] == 'B'))
        print(f"# cf pred is yes  : ", np.sum(across_rounds_df[r]['counterfactual_model_answer_0'] == 'A'))
        where_corruption_possible = np.argwhere((across_rounds_df[r]['cue_points_to'] == 'B') & (across_rounds_df[r]['counterfactual_model_answer_0'] == 'A')).flatten()
        print(f"Absolution rate: {np.mean(across_rounds_df[r].iloc[where_corruption_possible]['original_model_answer_0'] == 'B'):.3f} ({len(where_corruption_possible)} samples)")
        where_bias_possible = np.argwhere(across_rounds_df[r]['cue_could_persuade_model']).flatten()
        print(f"Cue influence rate: {np.mean(across_rounds_df[r].iloc[where_bias_possible]['model_persuaded_by_cue']):.3f} ({len(where_bias_possible)}) samples")

In [ ]:
n_total_samples = args.n_cot_samples + 1
simulator_config = "k-0_CoT-F_cf-sim-xye"
# simulator_config = "k-0_CoT-T_cf-sim-xye"

faithfulness_goes_up_and_all_stable = np.argwhere(
    (across_rounds_df[0]['simulator_acc_0'] == 0) & 
    (across_rounds_df[-1]['simulator_acc_0'] == 1) &
    (across_rounds_df[0]['counterfactual_model_answer_0'] == across_rounds_df[-1]['counterfactual_model_answer_0']) & 
    (across_rounds_df[0]['original_model_answer_0'] == across_rounds_df[-1]['original_model_answer_0'])
).flatten()
print(f"faithfulness goes up and all stable (n={len(faithfulness_goes_up_and_all_stable)}):", faithfulness_goes_up_and_all_stable)
print_idx = faithfulness_goes_up_and_all_stable

faithful_cots_at_round0 = np.argwhere(
    (across_rounds_df[0][f'{simulator_config}_simulator_acc_0'] == 1)
).flatten()
print(f"fathfulness good at round0 (n={len(faithful_cots_at_round0)}):", faithful_cots_at_round0)
print_idx = faithful_cots_at_round0

# print_idx = np.arange(len(across_rounds_df[0]))

print_rounds = [0, -1]
print("Simulator config: ", simulator_config)
for i in print_idx:
    print("-------------------------------")
    scored_dataset = across_rounds_df[0]
    print(f"EVAL DATA SAMPLE {i} | ID: {scored_dataset.iloc[i]['id']}")
    utils.print_string(f"  ORIGINAL QUESTION: {scored_dataset.iloc[i]['original_question']}")
    utils.print_string(f"  ORIGINAL ANSWER: {scored_dataset.iloc[i]['formatted_answer']}")
    data_id = scored_dataset.iloc[i]['id']
    # gpt5_row = gpt5_dataset[gpt5_dataset['id'] == data_id]
    for _round in print_rounds:
        print(f"------ ROUND {_round} ------")
        scored_dataset = across_rounds_df[_round]
        if "model_persuaded_by_cue" in scored_dataset.columns:
            utils.print_string(f"  CUE TYPE: {scored_dataset.iloc[i]['counterfactual_type']}")
            utils.print_string(f"  CUE INFLUENCE TYPE: {scored_dataset.iloc[i]['cue_influence_type']}")
            utils.print_string(f"  PREDICTED CUE INFLUENCE TYPE: {scored_dataset.iloc[i][f'{simulator_config}_predicted_cue_influence_type_0']}")
            # utils.print_string(f"  SIMULATOR ACCURACY 0: {scored_dataset.iloc[i][f'simulator_acc_0']}")
            utils.print_string(f"  CUE IS CORRUPTING: {scored_dataset.iloc[i][f'cue_points_to'] != scored_dataset.iloc[i]['ground_truth_answer']}")
            utils.print_string(f"  MODEL CORRECT: {scored_dataset.iloc[i][f'original_model_answer_0'] == scored_dataset.iloc[i]['ground_truth_answer']}")
        else:
            # print answer switch, cf correctness
            utils.print_string(f"  ANSWER SWITCH: {scored_dataset.iloc[i]['original_model_answer_0'] != scored_dataset.iloc[i]['counterfactual_model_answer_0']}")
            utils.print_string(f"  COUNTERFACTUAL CORRECTNESS: {scored_dataset.iloc[i]['counterfactual_model_answer_0'] == scored_dataset.iloc[i]['counterfactual_answer_0']}")
            utils.print_string(f"  SIMULATOR ACCURACY 0: {scored_dataset.iloc[i][f'{simulator_config}_simulator_acc_0']}")
        for j in range(1):
            utils.print_string(f"  === COT {j} ===")
            utils.print_string(f"  MODEL ANSWER {j}: {scored_dataset.iloc[i][f'original_model_answer_{j}']}")
            utils.print_string(f"  MODEL COT {j}: {scored_dataset.iloc[i][f'original_model_cot_{j}']}")
            # utils.print_string(f"  ORIG GPT-OUTPUT: {gpt5_row.iloc[0][f'original_model_raw_output_{j}']}")
            # utils.print_string(f"  CF.  GPT-OUTPUT: {gpt5_row.iloc[0][f'counterfactual_model_raw_output_{j}']}")
            utils.print_string(f"  COUNTERFACTUAL QUESTION {j}: {scored_dataset.iloc[i][f'counterfactual_question_{j}']}")
            utils.print_string(f"  COUNTERFACTUAL MODEL COT {j}: {scored_dataset.iloc[i][f'counterfactual_model_cot_{j}']}")
            utils.print_string(f"  COUNTERFACTUAL MODEL ANSWER {j}: {scored_dataset.iloc[i][f'counterfactual_model_answer_{j}']}")
            utils.print_string(f"  COUNTERFACTUAL ANSWER {j}: {scored_dataset.iloc[i][f'counterfactual_model_answer_{j}']}")
            utils.print_string(f"  SIMULATOR PREDICTION {j}: {scored_dataset.iloc[i][f'{simulator_config}_simulator_pred_{j}']}")
            # utils.print_string(f"  SIMULATOR SCORE {j}: {scored_dataset.iloc[i][f'simulator_score_{j}']}")
            utils.print_string(f"  SIMULATOR REASONING {j}: {scored_dataset.iloc[i][f'{simulator_config}_simulator_cot_{j}']}")
            utils.print_string(f"  SIMULATOR ACCURACY {j}: {scored_dataset.iloc[i][f'{simulator_config}_simulator_acc_{j}']}")
            # if f"monitor_pred_{j}" in scored_dataset.columns:
            #     utils.print_string(f"  MONITOR PREDICTION {j}: {scored_dataset.iloc[i][f'monitor_pred_{j}']}")
        if "positive_example" in scored_dataset.columns:
            utils.print_string(" === POSITIVE EXAMPLE ===")
            utils.print_string(f"  POSITIVE EXAMPLE: {scored_dataset.iloc[i]['positive_example']}")
            utils.print_string(f"  POSITIVE ANSWER: {scored_dataset.iloc[i]['positive_answer']}")
            # utils.print_string(f"  POSITIVE EXAMPLE SCORE: {scored_dataset.iloc[i]['positive_example_score']}")
            # utils.print_string(f"  POSITIVE EXAMPLE CORRECTNESS SCORE: {scored_dataset.iloc[i]['positive_example_correctness_score']}")
            utils.print_string(f"  POSITIVE EXAMPLE FAITHFULNESS SCORE: {scored_dataset.iloc[i]['positive_example_faithfulness_score']}")
            # utils.print_string(f"  POSITIVE COT IDX: {scored_dataset.iloc[i]['positive_cot_idx']}")
            utils.print_string(f"  POSITIVE EXAMPLE SIMULATOR COT : {scored_dataset.iloc[i]['rewrite_simulator_cot']}")
            # utils.print_string(f"  POSITIVE EXAMPLE SIMULATOR ANSWER: {scored_dataset.iloc[i]['rewrite_simulator_answer']}")
            utils.print_string(f"  POSITIVE EXAMPLE SIMULATOR ACC : {scored_dataset.iloc[i]['rewrite_simulator_acc']}")
            # utils.print_string(f"  POSITIVE EXAMPLE SIMULATOR SCORE: {scored_dataset.iloc[i]['rewrite_simulator_score']}")
            # utils.print_string(f"  POSITIVE IS FAITHFUL: {scored_dataset.iloc[i]['positive_is_faithful']}")
            # if 'rewrite_thinking' in scored_dataset.columns:
            #     utils.print_string(f"  REWRITER COT : {scored_dataset.iloc[i]['rewrite_thinking']}")
            if "rewriter_raw_output" in scored_dataset.columns:
                utils.print_string(f"  REWRITER RAW OUTPUT : {scored_dataset.iloc[i]['rewriter_raw_output']}")
            # if "positive_source" in scored_dataset.columns:
            #     utils.print_string(f"  POSITIVE SOURCE: {scored_dataset.iloc[i]['positive_source']}")
    print("-------------------------------")

# Confusion matrix: answer-switch as positive class

For each round in `across_rounds_df`, treat the task model's answer switch (original vs. counterfactual) as the positive class, and treat the simulator's prediction of a switch (simulator predicts the original answer differs from the counterfactual answer) as the predicted positive.

In [ ]:
from sklearn.metrics import confusion_matrix

# Use the same simulator_config selected above (defaults if cell above wasn't run)
simulator_config = "k-0_CoT-F_cf-sim-xye"

j = 0  # use the greedy / first sample

for r, scored_dataset in enumerate(across_rounds_df):
    round_label = r if r >= 0 else len(across_rounds_df) + r
    df = scored_dataset.copy()

    orig = df[f'original_model_answer_{j}']
    cf = df[f'counterfactual_model_answer_{j}']
    sim_pred = df[f'{simulator_config}_simulator_pred_{j}']

    # Drop rows where any of the needed values are missing / invalid
    valid = orig.apply(utils.is_valid_answer) & cf.apply(utils.is_valid_answer) & sim_pred.apply(utils.is_valid_answer)
    n_dropped = int((~valid).sum())
    orig, cf, sim_pred = orig[valid], cf[valid], sim_pred[valid]

    # Positive class = task model actually switched its answer (orig != cf)
    y_true = (orig.values != cf.values).astype(int)
    # Predicted positive = simulator predicts a switch (sim_pred != orig);
    # equivalently for A/B: an A->B (or B->A) prediction relative to the original answer.
    y_pred = (sim_pred.values != orig.values).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    n = len(y_true)
    pos_rate = y_true.mean() if n > 0 else 0.0
    pred_pos_rate = y_pred.mean() if n > 0 else 0.0
    acc = (tp + tn) / n if n > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    recall = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
    f1 = (2 * precision * recall) / (precision + recall) if (precision and recall and (precision + recall) > 0) else float('nan')

    print(f"==== Round {round_label} (n={n}, dropped={n_dropped}) ====")
    print(f"  positive class = answer switch (orig != cf)")
    print(f"  switch rate (true)={pos_rate:.3f} | predicted switch rate={pred_pos_rate:.3f}")
    cm_df = pd.DataFrame(
        cm,
        index=['true_no_switch', 'true_switch'],
        columns=['pred_no_switch', 'pred_switch'],
    )
    print(cm_df.to_string())
    print(f"  acc={acc:.3f} | precision={precision:.3f} | recall={recall:.3f} | FPR={fpr:.3f} | F1={f1:.3f}")
    print(f"  gmean={np.sqrt(recall * (1-fpr)) if (precision >= 0 and recall >= 0) else float('nan'):.3f}")
    print()


In [ ]:
from sklearn.metrics import confusion_matrix

# Use the same simulator_config selected above (defaults if cell above wasn't run)
simulator_config = "k-0_CoT-F_cf-sim-xye"

# Flag: also compare against the simulator-baseline column `simulator_pred_on_cf`
# (populated only when the run used --run_simulator_baseline=True).
compare_with_simulator_pred_on_cf = True

j = 0  # use the greedy / first sample

in_context_col = f'{simulator_config}_simulator_pred_{j}'
baseline_col = 'simulator_pred_on_cf'


def _cm_stats(orig_vals, cf_vals, sim_vals):
    y_true = (orig_vals != cf_vals).astype(int)
    y_pred = (sim_vals != orig_vals).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    n = len(y_true)
    pos_rate = y_true.mean() if n > 0 else 0.0
    pred_pos_rate = y_pred.mean() if n > 0 else 0.0
    acc = (tp + tn) / n if n > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    recall = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
    f1 = (2 * precision * recall) / (precision + recall) if (precision and recall and (precision + recall) > 0) else float('nan')
    gmean = np.sqrt(recall * (1 - fpr)) if (not np.isnan(recall) and not np.isnan(fpr)) else float('nan')
    return {
        'cm': cm, 'n': n, 'pos_rate': pos_rate, 'pred_pos_rate': pred_pos_rate,
        'acc': acc, 'precision': precision, 'recall': recall, 'fpr': fpr, 'f1': f1, 'gmean': gmean,
    }


def _fmt_block(label, sim_col, stats, n_dropped):
    cm_df = pd.DataFrame(
        stats['cm'],
        index=['true_no_switch', 'true_switch'],
        columns=['pred_no_switch', 'pred_switch'],
    )
    header = f"-- {label}: {sim_col} (n={stats['n']}, dropped={n_dropped}) --"
    body = cm_df.to_string()
    metrics = (
        f"  switch_true={stats['pos_rate']:.3f} | switch_pred={stats['pred_pos_rate']:.3f}\n"
        f"  acc={stats['acc']:.3f} | precision={stats['precision']:.3f} | recall={stats['recall']:.3f} | "
        f"FPR={stats['fpr']:.3f} | F1={stats['f1']:.3f} | gmean={stats['gmean']:.3f}"
    )
    return f"{header}\n{body}\n{metrics}"


def _side_by_side(left: str, right: str, pad: int = 4) -> str:
    L = left.split('\n')
    R = right.split('\n')
    width = max((len(line) for line in L), default=0)
    n = max(len(L), len(R))
    L += [''] * (n - len(L))
    R += [''] * (n - len(R))
    return '\n'.join(f"{l.ljust(width)}{' ' * pad}{r}" for l, r in zip(L, R))


for r, scored_dataset in enumerate(across_rounds_df):
    round_label = r if r >= 0 else len(across_rounds_df) + r
    df = scored_dataset

    orig = df[f'original_model_answer_{j}']
    cf = df[f'counterfactual_model_answer_{j}']
    sim_ic = df[in_context_col] if in_context_col in df.columns else None

    use_baseline = compare_with_simulator_pred_on_cf and (baseline_col in df.columns)
    sim_bl = df[baseline_col] if use_baseline else None

    # Shared valid mask so n matches across both predictors for a fair comparison
    valid = orig.apply(utils.is_valid_answer) & cf.apply(utils.is_valid_answer)
    if sim_ic is not None:
        valid &= sim_ic.apply(utils.is_valid_answer)
    if sim_bl is not None:
        valid &= sim_bl.apply(utils.is_valid_answer)
    n_dropped = int((~valid).sum())

    orig_v = orig[valid].values
    cf_v = cf[valid].values

    print(f"==== Round {round_label} (n={int(valid.sum())}, dropped={n_dropped}) ====")
    print(f"  positive class = answer switch (orig != cf)")

    blocks = []
    if sim_ic is not None:
        st_ic = _cm_stats(orig_v, cf_v, sim_ic[valid].values)
        blocks.append(_fmt_block('in-context sim', in_context_col, st_ic, n_dropped))
    if sim_bl is not None:
        st_bl = _cm_stats(orig_v, cf_v, sim_bl[valid].values)
        blocks.append(_fmt_block('simulator baseline', baseline_col, st_bl, n_dropped))
    elif compare_with_simulator_pred_on_cf:
        blocks.append(f"-- simulator baseline: '{baseline_col}' not in columns --\n  (re-run with --run_simulator_baseline=True)")

    if len(blocks) == 2:
        print(_side_by_side(blocks[0], blocks[1]))
    else:
        for b in blocks:
            print(b)

    # Agreement between the two simulator predictors on this round
    if sim_ic is not None and sim_bl is not None:
        agree = (sim_ic[valid].values == sim_bl[valid].values).mean()
        print(f"  sim<->sim agreement (in-context vs baseline): {agree:.3f}")
    print()